# Retinal Vessel Segmentation + Artery/Vein Classifier

Two-stage pipeline:
1. **Binary Segmentation** – MorphoBiFPNUNet: background vs vessel (trained on 8-image subset)
2. **AV Classifier** – Lightweight CNN trained on the **full training set** to label artery vs vein

**Dataset layout** (each split folder):
```
Fundus-AVSeg-prep-data/
  train/    images/, annotation/, npy/
  val/      images/, annotation/, npy/
  test/     images/, annotation/, npy/
```

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as T
import math
import cv2
from pathlib import Path
import skimage
from skimage.measure import label, regionprops
import copy, os, time
from collections import Counter

print("All imports successful.")

## Constants

In [ ]:
# ── Binary segmentation constants ─────────────────────────────────────────────
NUM_CLASSES_SEG  = 2          # 0=background, 1=vessel
SEG_CLASS_NAMES  = {0: 'Background', 1: 'Vessel'}
SEG_CLASS_COLORS = {
    0: np.array([  0,   0,   0], dtype=np.uint8),   # Background: black
    1: np.array([255, 255,   0], dtype=np.uint8),   # Vessel:     yellow
}

# ── AV classifier constants ───────────────────────────────────────────────────
AV_CLASS_NAMES   = {1: 'Artery', 2: 'Vein'}        # original label indices
AV_CLASS_COLORS  = {
    1: np.array([255,   0,   0], dtype=np.uint8),   # Artery: red
    2: np.array([  0,   0, 255], dtype=np.uint8),   # Vein:   blue
}

# ── Training subset ───────────────────────────────────────────────────────────
FIXED_INDICES = [1, 3, 5, 6, 7, 12, 15, 16]

print(f"Binary segmentation: {NUM_CLASSES_SEG} classes (background vs vessel)")
print(f"AV classifier: artery vs vein")
print(f"Training subset indices: {FIXED_INDICES}")

## Dataset Classes

In [ ]:
class BinarySegDataset(data.Dataset):
    """
    Binary segmentation dataset: background (0) vs vessel (1).
    Arteries and veins are both mapped to vessel class 1.
    Crossing and Uncertainty pixels are remapped to Background.
    """

    def __init__(self, root, new_size=592, supervised=True):
        self.root       = Path(root)
        self.supervised = supervised
        self.new_size   = new_size
        self.imagesize  = None

        img_dir = self.root / 'images'
        npy_dir = self.root / 'npy'
        ann_dir = self.root / 'annotation'

        self.image_paths         = sorted(img_dir.glob('*.png'))
        self.image_liot_eg_paths = []
        self.label_paths         = []

        for img_path in self.image_paths:
            stem = img_path.stem
            self.image_liot_eg_paths.append(npy_dir / f'{stem}_liot_eg.npy')
            if self.supervised:
                self.label_paths.append(ann_dir / f'{stem}.png')

        print(f"Dataset root     : {root}")
        print(f"  Images found   : {len(self.image_paths)}")

    def borders(self, original_size):
        H, W    = original_size
        delta_h = max(self.new_size - H, 0)
        delta_w = max(self.new_size - W, 0)
        top     = delta_h // 2
        bottom  = delta_h - top
        left    = delta_w // 2
        right   = delta_w - left
        return left, top, right, bottom

    @staticmethod
    def rgb_to_binary_mask(rgb_array):
        """
        Convert RGB annotation to binary mask:
          0 = background (black, green, white)
          1 = vessel     (red artery OR blue vein)
        """
        r, g, b = rgb_array[:, :, 0], rgb_array[:, :, 1], rgb_array[:, :, 2]
        mask    = np.zeros(rgb_array.shape[:2], dtype=np.int64)
        # Artery (red-dominant) → vessel
        mask[(r > 180) & (g < 80)  & (b < 80)]  = 1
        # Vein   (blue-dominant) → vessel
        mask[(r < 80)  & (g < 80)  & (b > 180)] = 1
        return mask

    @staticmethod
    def rgb_to_av_mask(rgb_array):
        """
        Convert RGB annotation to AV class mask:
          0 = background
          1 = artery
          2 = vein
        (Used by classifier dataset to generate AV ground truth.)
        """
        r, g, b = rgb_array[:, :, 0], rgb_array[:, :, 1], rgb_array[:, :, 2]
        mask    = np.zeros(rgb_array.shape[:2], dtype=np.int64)
        mask[(r > 180) & (g < 80)  & (b < 80)]  = 1  # artery
        mask[(r < 80)  & (g < 80)  & (b > 180)] = 2  # vein
        return mask

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert('RGB')
        self.imagesize        = np.array(image).shape[:2]
        left, top, right, bottom = self.borders(self.imagesize)
        pad   = (left, top, right, bottom)

        img_tf = T.Compose([T.ToTensor(), T.Pad(pad)])
        image  = img_tf(image)

        liot = np.load(self.image_liot_eg_paths[index])
        liot = img_tf(liot.transpose(1, 2, 0))

        if self.supervised:
            ann_rgb   = np.array(Image.open(self.label_paths[index]).convert('RGB'))
            bin_mask  = self.rgb_to_binary_mask(ann_rgb)
            av_mask   = self.rgb_to_av_mask(ann_rgb)
            bin_mask  = np.pad(bin_mask, ((top, bottom), (left, right)), constant_values=0)
            av_mask   = np.pad(av_mask,  ((top, bottom), (left, right)), constant_values=0)
            bin_label = torch.from_numpy(bin_mask)
            av_label  = torch.from_numpy(av_mask)
            return image, liot, bin_label, av_label

        return image, liot

## Load Data

In [ ]:
root = r"/home/mefeki/Bureau/work/Fundus-AVSeg-prep-data"   # ← update if needed

train_dataset_full = BinarySegDataset(root=os.path.join(root, 'train'))
val_dataset        = BinarySegDataset(root=os.path.join(root, 'val'))
test_dataset       = BinarySegDataset(root=os.path.join(root, 'test'))

val_dataloader  = data.DataLoader(val_dataset,  shuffle=False, batch_size=1)
test_dataloader = data.DataLoader(test_dataset, shuffle=False, batch_size=1)

print(f"Train (full): {len(train_dataset_full)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## Utilities

In [ ]:
def bin_mask_to_rgb(mask_np):
    """Convert binary (H,W) int mask → RGB (H,W,3)."""
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for cls_idx, color in SEG_CLASS_COLORS.items():
        rgb[mask_np == cls_idx] = color
    return rgb


def av_mask_to_rgb(mask_np):
    """Convert AV (H,W) mask (0=bg,1=artery,2=vein) → RGB (H,W,3)."""
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    rgb[mask_np == 1] = AV_CLASS_COLORS[1]   # artery: red
    rgb[mask_np == 2] = AV_CLASS_COLORS[2]   # vein:   blue
    return rgb


def remove_circle(inputs, preds, kernel_size=7, threshold=0.1):
    """Zero-out predictions outside the circular FOV."""
    if inputs.shape[1] == 4:
        rgb   = inputs[:, :3]
        alpha = inputs[:, 3:]
        inputs = rgb * alpha + (1 - alpha)
    gray = T.Grayscale()(inputs)
    mask = (gray > threshold).float()
    p1 = -F.max_pool2d(-mask, (kernel_size, 1), stride=1, padding=(kernel_size // 2, 0))
    p2 = -F.max_pool2d(-mask, (1, kernel_size), stride=1, padding=(0, kernel_size // 2))
    circle = (torch.min(p1, p2) > 0).squeeze(1).long()
    return preds * circle


def dice_binary(preds, labels, eps=1e-6):
    """Dice for vessel class (class 1)."""
    p = (preds == 1).float()
    l = (labels == 1).float()
    tp = float(torch.sum(p * l))
    fp = float(torch.sum(p * (1 - l)))
    fn = float(torch.sum((1 - p) * l))
    return 2 * tp / (2 * tp + fp + fn + eps)


device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Network Architecture: MorphoBiFPNUNet (Binary)

In [ ]:
# ── Shared building blocks ──────────────────────────────────────────────────

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.conv(x)


class MorphoLayer(nn.Module):
    """
    Learnable morphological pre-processing:
      dilation = max_pool2d(x, k)
      erosion  = -max_pool2d(-x, k)
    Concatenate both -> project back to in_ch via 1x1 conv.
    """
    def __init__(self, in_ch, kernel_size=7):
        super().__init__()
        self.k    = kernel_size
        self.proj = nn.Sequential(
            nn.Conv2d(in_ch * 2, in_ch, 1, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        pad      = self.k // 2
        dilation = F.max_pool2d(x,  self.k, stride=1, padding=pad)
        erosion  = -F.max_pool2d(-x, self.k, stride=1, padding=pad)
        return self.proj(torch.cat([dilation, erosion], dim=1))


class Encoder(nn.Module):
    def __init__(self, in_ch=10, base_ch=64, depth=4):
        super().__init__()
        chs = [base_ch * (2 ** i) for i in range(depth)]
        self.blocks = nn.ModuleList()
        self.pools  = nn.ModuleList()
        in_c = in_ch
        for c in chs:
            self.blocks.append(ConvBlock(in_c, c))
            self.pools.append(nn.MaxPool2d(2))
            in_c = c
        self.out_channels = chs

    def forward(self, x):
        feats = []
        for block, pool in zip(self.blocks, self.pools):
            x = block(x)
            feats.append(x)
            x = pool(x)
        return feats


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))


class BiFPN(nn.Module):
    """
    Bidirectional FPN with fast-normalised fusion.

    Fix: all encoder levels are projected to a uniform feat_ch via lateral
    1x1 convs BEFORE weighted fusion, so tensors of different channel widths
    (e.g. 64, 128, 256, 512) are never added together directly.
    """
    def __init__(self, channels, feat_ch=256, num_layers=2, eps=1e-4):
        super().__init__()
        self.eps        = eps
        self.feat_ch    = feat_ch
        self.num_levels = len(channels)
        self.num_layers = num_layers

        # Project every encoder level to feat_ch
        self.laterals = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, feat_ch, 1, bias=False),
                nn.BatchNorm2d(feat_ch),
                nn.ReLU(inplace=True),
            ) for c in channels
        ])

        # All fusion convs now operate at the same feat_ch
        self.convs_td = nn.ModuleList([ConvBlock(feat_ch, feat_ch) for _ in channels])
        self.convs_bu = nn.ModuleList([ConvBlock(feat_ch, feat_ch) for _ in channels])
        self.w_td     = nn.Parameter(torch.ones(num_layers, self.num_levels, 2))
        self.w_bu     = nn.Parameter(torch.ones(num_layers, self.num_levels, 3))

    def _resize(self, src, tgt):
        if src.shape[-2:] != tgt.shape[-2:]:
            return F.interpolate(src, size=tgt.shape[-2:], mode='bilinear', align_corners=False)
        return src

    def forward(self, feats):
        # Project all levels to feat_ch first
        P = [lat(f) for lat, f in zip(self.laterals, feats)]

        for l in range(self.num_layers):
            td = [None] * self.num_levels
            td[-1] = P[-1]
            w_td   = F.relu(self.w_td[l])
            w_td   = w_td / (w_td.sum(-1, keepdim=True) + self.eps)
            for i in range(self.num_levels - 2, -1, -1):
                fused = w_td[i, 0] * P[i] + w_td[i, 1] * self._resize(td[i + 1], P[i])
                td[i] = self.convs_td[i](fused)

            bu = [None] * self.num_levels
            bu[0] = td[0]
            w_bu  = F.relu(self.w_bu[l])
            w_bu  = w_bu / (w_bu.sum(-1, keepdim=True) + self.eps)
            for i in range(1, self.num_levels):
                fused = (w_bu[i, 0] * P[i] +
                         w_bu[i, 1] * td[i] +
                         w_bu[i, 2] * self._resize(bu[i - 1], P[i]))
                bu[i] = self.convs_bu[i](fused)
            P = bu
        return P


class MorphoBiFPNUNet(nn.Module):
    """
    MorphoBiFPNUNet for binary segmentation (background vs vessel).
    The decoder uses feat_ch (BiFPN output width) as its skip channel size.
    """
    def __init__(self, in_channels=10, num_classes=2, base_ch=64,
                 depth=4, bifpn_layers=2, morpho_kernel=7, bifpn_feat_ch=256):
        super().__init__()
        self.morpho     = MorphoLayer(in_channels, kernel_size=morpho_kernel)
        self.encoder    = Encoder(in_ch=in_channels, base_ch=base_ch, depth=depth)
        enc_chs         = self.encoder.out_channels          # e.g. [64,128,256,512]
        self.bifpn      = BiFPN(channels=enc_chs, feat_ch=bifpn_feat_ch, num_layers=bifpn_layers)
        self.bottleneck = ConvBlock(bifpn_feat_ch, bifpn_feat_ch)

        # Decoder: every level now has bifpn_feat_ch channels after BiFPN
        self.up_blocks = nn.ModuleList()
        in_c = bifpn_feat_ch
        for _ in range(depth - 1):
            self.up_blocks.append(UpBlock(in_c, bifpn_feat_ch, bifpn_feat_ch))
            in_c = bifpn_feat_ch
        self.final_conv = nn.Conv2d(in_c, num_classes, 1)

    def forward(self, x):
        x     = self.morpho(x)
        feats = self.encoder(x)
        fused = self.bifpn(feats)          # all levels now bifpn_feat_ch wide
        x     = self.bottleneck(fused[-1])
        for i, up in enumerate(self.up_blocks):
            x = up(x, fused[-2 - i])
        return self.final_conv(x)


print('MorphoBiFPNUNet (binary, fixed BiFPN) defined.')


## Loss & Training / Validation Functions

In [ ]:
class WeightedCELoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, outputs, labels):
        return self.criterion(outputs, labels.long())


def train_seg_model(model, criterion, optimizer, dataloader, num_epochs, val_loader=None):
    """Train binary segmentation model; validate each epoch on val_loader."""
    train_losses = []
    val_dices    = []

    for epoch in range(num_epochs):
        model.train()
        batch_losses = []

        for inputs, liot, bin_lbl, _ in dataloader:
            liot    = liot.to(device)
            bin_lbl = bin_lbl.to(device)

            optimizer.zero_grad()
            logits = model(liot)
            loss   = criterion(logits, bin_lbl)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        mean_loss = sum(batch_losses) / len(batch_losses)
        train_losses.append(mean_loss)

        val_dice = 0.0
        if val_loader is not None:
            model.eval()
            with torch.no_grad():
                dices = []
                for img, liot_v, bin_lbl, _ in val_loader:
                    img     = img.to(device)
                    liot_v  = liot_v.to(device)
                    bin_lbl = bin_lbl.to(device)
                    logits  = model(liot_v)
                    preds   = torch.argmax(torch.softmax(logits, dim=1), dim=1)
                    preds   = remove_circle(img, preds)
                    dices.append(dice_binary(preds, bin_lbl))
                val_dice = float(np.mean(dices))
            val_dices.append(val_dice)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {mean_loss:.4f}"
                  + (f" | Val Dice: {val_dice:.4f}" if val_loader else ""))

    print("Training complete.")
    return model, train_losses, val_dices


print("Loss and training functions defined.")

## Train Binary Segmentation Model (8-image subset)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

# ── Fixed 8-image subset ──────────────────────────────────────────────────────
assert all(i < len(train_dataset_full) for i in FIXED_INDICES), \
    f"Index out of range (max={len(train_dataset_full)-1})"

subset_dataset    = torch.utils.data.Subset(train_dataset_full, FIXED_INDICES)
subset_dataloader = data.DataLoader(subset_dataset, shuffle=True, batch_size=1)

print(f"Training on {len(subset_dataset)} images (indices {FIXED_INDICES})")
print(f"Validation on {len(val_dataset)} images")

# ── Class weights from subset ─────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

pixel_counts = Counter()
for _, _, bin_lbl, _ in subset_dataset:
    for c in range(NUM_CLASSES_SEG):
        pixel_counts[c] += int((bin_lbl == c).sum())

total    = sum(pixel_counts.values())
freqs    = torch.tensor([pixel_counts[c] / total for c in range(NUM_CLASSES_SEG)])
inv_freq = 1.0 / (freqs + 1e-6)
class_weights = (inv_freq / inv_freq.mean()).to(device)

print(f"Class pixel %: background={100*freqs[0]:.2f}%, vessel={100*freqs[1]:.2f}%")
print(f"Class weights: {class_weights.tolist()}")

# ── Model, loss, optimiser ────────────────────────────────────────────────────
EPOCHS = 150

seg_model = MorphoBiFPNUNet(
    in_channels=10, num_classes=NUM_CLASSES_SEG, morpho_kernel=7
).to(device)

seg_criterion = WeightedCELoss(class_weights=class_weights)
seg_optimizer = optim.Adam(seg_model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-7)

seg_model, train_losses, val_dices = train_seg_model(
    seg_model, seg_criterion, seg_optimizer,
    subset_dataloader, EPOCHS, val_loader=val_dataloader
)

# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)
torch.save(seg_model.state_dict(), 'models/seg_model_binary.pth')
print("Segmentation model saved to models/seg_model_binary.pth")

## Segmentation Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS + 1), train_losses, color='steelblue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('WCE Loss')
axes[0].set_title('Training Loss – Binary Segmentation')
axes[0].grid(True)

if val_dices:
    axes[1].plot(range(1, EPOCHS + 1), val_dices, color='coral')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice (vessel)')
    axes[1].set_title(f'Validation Dice – Best: {max(val_dices):.4f}')
    axes[1].grid(True)

plt.tight_layout()
plt.savefig('seg_training_curves.png', dpi=150)
plt.show()

## AV Classifier Architecture

In [ ]:
class AVClassifier(nn.Module):
    """
    Lightweight pixel-wise AV classifier.

    Takes the RGB fundus image and a binary vessel mask as input
    (4 channels total) and produces a 2-class output map:
        0 → artery
        1 → vein

    Only vessel pixels (mask==1) are classified; background pixels
    are ignored during training and forced to 0 during inference.

    Architecture: simple FCN with 3 ConvBlocks + 1×1 head.
    """
    def __init__(self, in_ch=4, base_ch=32, num_av=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch), nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch * 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch * 2), nn.ReLU(inplace=True),
            nn.Conv2d(base_ch * 2, base_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch), nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, num_av, 1),
        )

    def forward(self, x):
        return self.net(x)   # (B, 2, H, W) logits


print("AVClassifier defined.")

## AV Classifier Dataset

In [ ]:
class AVClassifierDataset(data.Dataset):
    """
    Wraps BinarySegDataset to feed the AV classifier.

    Returns:
        x        : (4, H, W) = cat([rgb_image, binary_vessel_mask_float])
        av_label : (H, W)    int64, values in {0 (artery), 1 (vein)}
                              (valid only where binary_mask==1)
        bin_mask : (H, W)    0/1 LongTensor — vessel mask
    """
    def __init__(self, seg_dataset):
        self.dataset = seg_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _, bin_mask, av_mask = self.dataset[idx]
        # Stack rgb + vessel mask float as input
        vessel_float = bin_mask.float().unsqueeze(0)       # (1, H, W)
        x = torch.cat([img, vessel_float], dim=0)          # (4, H, W)

        # AV label: artery→0, vein→1 (classifier output space)
        av_cls = (av_mask - 1).clamp(min=0)                # 1→0, 2→1
        return x, av_cls.long(), bin_mask.long()


# Build AV datasets from all 3 splits
# Classifier is trained on the FULL training set (all available annotated images)
av_train_dataset = AVClassifierDataset(train_dataset_full)
av_val_dataset  = AVClassifierDataset(val_dataset)
av_test_dataset = AVClassifierDataset(test_dataset)

av_train_loader = data.DataLoader(av_train_dataset, shuffle=True,  batch_size=1)
av_val_loader   = data.DataLoader(av_val_dataset,  shuffle=False, batch_size=1)
av_test_loader  = data.DataLoader(av_test_dataset, shuffle=False, batch_size=1)

print(f"AV classifier datasets – Train: {len(av_train_dataset)}, "
      f"Val: {len(av_val_dataset)}, Test: {len(av_test_dataset)}")

## Train AV Classifier

In [ ]:
def train_av_classifier(model, optimizer, loader, num_epochs, val_loader=None):
    """
    Train the AV classifier with masked cross-entropy
    (only vessel pixels participate in the loss).
    """
    criterion = nn.CrossEntropyLoss(reduction='none')
    train_losses = []
    val_accs     = []

    for epoch in range(num_epochs):
        model.train()
        batch_losses = []

        for x, av_lbl, bin_mask in loader:
            x, av_lbl, bin_mask = x.to(device), av_lbl.to(device), bin_mask.to(device)

            optimizer.zero_grad()
            logits     = model(x)                             # (B,2,H,W)
            pixel_loss = criterion(logits, av_lbl)            # (B,H,W)
            vessel_px  = bin_mask.float()
            loss       = (pixel_loss * vessel_px).sum() / (vessel_px.sum() + 1e-6)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        mean_loss = sum(batch_losses) / len(batch_losses)
        train_losses.append(mean_loss)

        val_acc = 0.0
        if val_loader is not None:
            model.eval()
            with torch.no_grad():
                total_correct = 0; total_vessel = 0
                for x, av_lbl, bin_mask in val_loader:
                    x, av_lbl, bin_mask = x.to(device), av_lbl.to(device), bin_mask.to(device)
                    preds   = torch.argmax(model(x), dim=1)  # (B,H,W): 0=artery,1=vein
                    vessel  = (bin_mask == 1)
                    correct = ((preds == av_lbl) & vessel).sum().item()
                    total_correct += correct
                    total_vessel  += vessel.sum().item()
                val_acc = total_correct / (total_vessel + 1e-6)
            val_accs.append(val_acc)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {mean_loss:.4f}"
                  + (f" | Val Acc: {val_acc:.4f}" if val_loader else ""))

    print("AV classifier training complete.")
    return model, train_losses, val_accs


torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

AV_EPOCHS = 80
av_model     = AVClassifier(in_ch=4, base_ch=32, num_av=2).to(device)
av_optimizer = optim.Adam(av_model.parameters(), lr=1e-3)

av_model, av_losses, av_val_accs = train_av_classifier(
    av_model, av_optimizer, av_train_loader,
    AV_EPOCHS, val_loader=av_val_loader
)

torch.save(av_model.state_dict(), 'models/av_classifier.pth')
print("AV classifier saved to models/av_classifier.pth")

## AV Classifier Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, AV_EPOCHS + 1), av_losses, color='teal')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('AV Classifier – Training Loss')
axes[0].grid(True)

if av_val_accs:
    axes[1].plot(range(1, AV_EPOCHS + 1), av_val_accs, color='darkorange')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Pixel Accuracy (vessel)')
    axes[1].set_title(f'AV Classifier – Val Accuracy (best {max(av_val_accs):.4f})')
    axes[1].grid(True)

plt.tight_layout()
plt.savefig('av_classifier_curves.png', dpi=150)
plt.show()

## Two-Stage Inference Pipeline

```
LIOT  →  MorphoBiFPNUNet  →  binary_mask (vessel / background)
RGB + binary_mask  →  AVClassifier  →  artery / vein map
```

In [ ]:
def run_pipeline(seg_model, av_model, img_t, liot_t):
    """
    Full two-stage inference.

    Parameters
    ----------
    seg_model : MorphoBiFPNUNet (binary)
    av_model  : AVClassifier
    img_t     : (1, 3, H, W) float tensor, RGB fundus image
    liot_t    : (1, 10, H, W) float tensor, LIOT features

    Returns
    -------
    bin_pred  : (H, W) LongTensor – 0=background, 1=vessel
    av_pred   : (H, W) LongTensor – 0=background, 1=artery, 2=vein
    """
    seg_model.eval(); av_model.eval()

    with torch.no_grad():
        # Stage 1: binary segmentation
        logits   = seg_model(liot_t)
        bin_pred = torch.argmax(torch.softmax(logits, dim=1), dim=1)  # (1,H,W)
        bin_pred = remove_circle(img_t, bin_pred)                     # (1,H,W)

        # Stage 2: AV classification on vessel pixels only
        x_cls    = torch.cat([img_t, bin_pred.float().unsqueeze(1)], dim=1)  # (1,4,H,W)
        av_logits = av_model(x_cls)                                          # (1,2,H,W)
        av_raw    = torch.argmax(av_logits, dim=1)                           # (1,H,W): 0=artery,1=vein

        # Map back to label space: artery=1, vein=2
        av_out    = (av_raw + 1) * bin_pred   # 0 where no vessel, 1=artery, 2=vein

    return bin_pred[0].cpu(), av_out[0].cpu()


print("Pipeline function defined.")

## Evaluate Full Pipeline on Test Set

In [ ]:
def evaluate_pipeline(seg_model, av_model, test_ds, n_display=4):
    """
    Run and visualise the pipeline on n_display test images.
    Reports binary vessel Dice and AV pixel accuracy.
    """
    seg_dices  = []
    av_accs    = []
    artery_acc = []
    vein_acc   = []

    indices = list(range(len(test_ds)))
    display_idx = random.sample(indices, min(n_display, len(indices)))

    all_results = []

    for i in range(len(test_ds)):
        img_t, liot_t, bin_gt, av_gt = test_ds[i]
        img_t  = img_t.unsqueeze(0).to(device)
        liot_t = liot_t.unsqueeze(0).to(device)
        bin_gt = bin_gt.to(device)
        av_gt  = av_gt.to(device)

        bin_pred, av_pred = run_pipeline(seg_model, av_model, img_t, liot_t)
        bin_pred = bin_pred.to(device)
        av_pred  = av_pred.to(device)

        # ── Metrics ──────────────────────────────────────────────────────────
        seg_dices.append(dice_binary(bin_pred, bin_gt))

        # AV accuracy: over vessel pixels only
        vessel_mask = (bin_gt == 1)
        if vessel_mask.sum() > 0:
            av_correct  = ((av_pred == av_gt) & vessel_mask).sum().item()
            av_accs.append(av_correct / vessel_mask.sum().item())

            # Per-class
            art_mask = (av_gt == 1)
            vei_mask = (av_gt == 2)
            if art_mask.sum() > 0:
                artery_acc.append(((av_pred == 1) & art_mask).sum().item() / art_mask.sum().item())
            if vei_mask.sum() > 0:
                vein_acc.append(((av_pred == 2) & vei_mask).sum().item() / vei_mask.sum().item())

        all_results.append((i, img_t.cpu(), bin_pred.cpu(), av_pred.cpu(),
                            bin_gt.cpu(), av_gt.cpu(),
                            seg_dices[-1], av_accs[-1] if av_accs else 0))

    print("\n=== Test Set Evaluation ===")
    print(f"Binary Vessel Dice        : {np.mean(seg_dices):.4f} ± {np.std(seg_dices):.4f}")
    print(f"AV Pixel Accuracy         : {np.mean(av_accs):.4f} ± {np.std(av_accs):.4f}")
    print(f"  Artery pixel accuracy   : {np.mean(artery_acc):.4f}")
    print(f"  Vein   pixel accuracy   : {np.mean(vein_acc):.4f}")

    # ── Visualise n_display examples ─────────────────────────────────────────
    print(f"\nShowing {len(display_idx)} test examples…")
    for i in display_idx:
        _, img_t, bin_pred, av_pred, bin_gt, av_gt, d, acc = all_results[i]
        img_np    = img_t[0].permute(1, 2, 0).clamp(0, 1).numpy()

        # Convert masks to RGB
        bin_pred_rgb = bin_mask_to_rgb(bin_pred.numpy())
        bin_gt_rgb   = bin_mask_to_rgb(bin_gt.numpy())
        av_pred_rgb  = av_mask_to_rgb(av_pred.numpy())
        av_gt_rgb    = av_mask_to_rgb(av_gt.numpy())

        fig, axes = plt.subplots(1, 5, figsize=(20, 4))
        axes[0].imshow(img_np);         axes[0].set_title('Fundus Image')
        axes[1].imshow(bin_gt_rgb);     axes[1].set_title('Vessel GT (binary)')
        axes[2].imshow(bin_pred_rgb);   axes[2].set_title(f'Vessel Pred  Dice={d:.3f}')
        axes[3].imshow(av_gt_rgb);      axes[3].set_title('AV Ground Truth')
        axes[4].imshow(av_pred_rgb);    axes[4].set_title(f'AV Prediction  Acc={acc:.3f}')

        for ax in axes: ax.axis('off')

        # Legend for AV maps
        legend = [
            mpatches.Patch(color=[1, 0, 0], label='Artery'),
            mpatches.Patch(color=[0, 0, 1], label='Vein'),
        ]
        axes[4].legend(handles=legend, loc='lower right', fontsize=8)
        fig.suptitle(f"Test image #{i}", fontsize=12)
        fig.tight_layout()
        plt.savefig(f'test_result_{i}.png', dpi=150)
        plt.show()

    return seg_dices, av_accs


seg_dices, av_accs = evaluate_pipeline(
    seg_model, av_model, test_dataset, n_display=4
)

## Summary Table

In [ ]:
summary = pd.DataFrame({
    'Test Image': [f'#{i}' for i in range(len(seg_dices))],
    'Vessel Dice': [f'{d:.4f}' for d in seg_dices],
    'AV Pixel Acc': [f'{a:.4f}' for a in (av_accs if len(av_accs) == len(seg_dices) else av_accs + [None] * (len(seg_dices) - len(av_accs)))],
})
summary.loc[len(summary)] = ['Mean', f'{np.mean(seg_dices):.4f}', f'{np.mean(av_accs):.4f}']
print(summary.to_string(index=False))